# Bedrock AgentCore Gateway를 사용하여 OpenAPI API를 MCP 도구로 변환하기

## 개요
고객은 JSON 또는 YAML 형식의 OpenAPI 사양을 가져와 Bedrock AgentCore Gateway를 사용해 API를 MCP 도구로 변환할 수 있습니다. 여기서는 API 키를 사용하여 NASA Open APIs를 호출하는 화성 기상 에이전트를 구축하는 방법을 살펴봅니다. 

Gateway 워크플로에서 에이전트를 외부 도구에 연결하는 단계는 다음과 같습니다.
* **Gateway용 도구 생성** - REST API의 OpenAPI 사양과 같은 스키마를 사용하여 도구를 정의합니다. 그러면 Amazon Bedrock AgentCore가 OpenAPI 사양을 파싱하여 Gateway를 생성합니다.
* **Gateway 엔드포인트 생성** - 인바운드 인증을 사용하는 MCP 진입점 역할의 Gateway를 생성합니다.
* **Gateway에 대상 추가** - Gateway가 요청을 특정 도구로 라우팅하는 방법을 정의하는 OpenAPI 대상을 구성합니다. OpenAPI 파일에 포함된 모든 API는 MCP 호환 도구로 변환되어 Gateway 엔드포인트 URL을 통해 제공됩니다. 각 OpenAPI Gateway 대상의 아웃바운드 권한 부여를 구성합니다. 
* **에이전트 코드 업데이트** - 에이전트를 Gateway 엔드포인트에 연결하여 통합 MCP 인터페이스를 통해 구성된 모든 도구에 액세스합니다.

![작동 방식](images/openapi-gateway-apikey.png)

### 튜토리얼 세부 정보


| 정보                 | 세부 정보                                                 |
|:---------------------|:----------------------------------------------------------|
| 튜토리얼 유형         | 대화형                                                     |
| AgentCore 구성 요소  | AgentCore Gateway, AgentCore Identity                     |
| 에이전트 프레임워크   | Strands Agents                                            |
| Gateway 대상 유형    | OpenAPI                                                   |
| 에이전트              | 화성 기상 에이전트                                        |
| 인바운드 인증 IdP     | Amazon Cognito                                            |
| 아웃바운드 인증       | API 키                                                    |
| LLM 모델              | Anthropic Claude Haiku 4.5, Amazon Nova Pro              |
| 튜토리얼 구성 요소    | AgentCore Gateway 생성 및 호출                            |
| 튜토리얼 분야         | 산업 전반                                                 |
| 예제 난이도           | 쉬움                                                      |
| 사용 SDK              | boto3                                                     |

튜토리얼의 첫 번째 부분에서는 몇 가지 AmazonCore Gateway 대상을 생성합니다.

### 튜토리얼 아키텍처
이 튜토리얼에서는 OpenAPI yaml/json 파일에 정의된 작업을 MCP 도구로 변환하고 Bedrock AgentCore Gateway에서 호스팅합니다.
데모를 위해 화성 날씨 관련 질문에 답하는 화성 기상 에이전트를 구축합니다. 이 에이전트는 NASA Open APIs를 사용합니다. 이 솔루션에서는 Amazon Bedrock 모델을 사용하는 Strands Agent를 활용합니다.
이 예제에서는 화성 날씨용 getInsightWeather 도구를 사용하는 매우 간단한 에이전트를 활용합니다.

## 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Jupyter Notebook(Python 커널)
* uv
* AWS 자격 증명
* Amazon Cognito

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
# Amazon SageMaker Notebook을 사용하지 않는 경우 AWS 자격 증명 설정
import os

# os.environ['AWS_ACCESS_KEY_ID'] = '' # Set the access key
# os.environ['AWS_SECRET_ACCESS_KEY'] = '' # Set the secret key
os.environ["AWS_DEFAULT_REGION"] = os.environ.get("AWS_REGION", "us-east-1")

In [ ]:
import os
import sys

# 현재 스크립트의 디렉터리 확인
if "__file__" in globals():
    current_dir = os.path.dirname(os.path.abspath(__file__))
else:
    current_dir = os.getcwd()  # __file__이 정의되지 않은 경우(예: Jupyter) 대체 경로 사용

# utils.py가 있는 디렉터리로 이동(한 단계 상위)
utils_dir = os.path.abspath(os.path.join(current_dir, "../.."))

# sys.path에 추가
sys.path.insert(0, utils_dir)

# 이제 utils를 가져올 수 있음
import utils

In [ ]:
#### Gateway가 수임할 IAM 역할 생성

agentcore_gateway_iam_role = utils.create_agentcore_gateway_role("sample-lambdagateway")
print("Agentcore gateway role ARN: ", agentcore_gateway_iam_role["Role"]["Arn"])

# Gateway 인바운드 권한 부여를 위한 Amazon Cognito 풀 생성

In [ ]:
# Cognito 사용자 풀 생성
import os
import boto3

REGION = os.environ["AWS_DEFAULT_REGION"]
USER_POOL_NAME = "sample-agentcore-gateway-pool"
RESOURCE_SERVER_ID = "sample-agentcore-gateway-id"
RESOURCE_SERVER_NAME = "sample-agentcore-gateway-name"
CLIENT_NAME = "sample-agentcore-gateway-client"
SCOPES = [
    {"ScopeName": "gateway:read", "ScopeDescription": "Read access"},
    {"ScopeName": "gateway:write", "ScopeDescription": "Write access"},
]
scopeString = f"{RESOURCE_SERVER_ID}/gateway:read {RESOURCE_SERVER_ID}/gateway:write"

cognito = boto3.client("cognito-idp", region_name=REGION)

print("Creating or retrieving Cognito resources...")
user_pool_id = utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
print(f"User Pool ID: {user_pool_id}")

utils.get_or_create_resource_server(cognito, user_pool_id, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES)
print("Resource server ensured.")

client_id, client_secret = utils.get_or_create_m2m_client(cognito, user_pool_id, CLIENT_NAME, RESOURCE_SERVER_ID)
print(f"Client ID: {client_id}")

# 디스커버리 URL 확인
cognito_discovery_url = f"https://cognito-idp.{REGION}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration"
print(cognito_discovery_url)

# Gateway 생성

In [ ]:
# CMK 없이 Cognito 권한 부여자를 사용하여 CreateGateway 호출. 이전 단계에서 생성한 Cognito 사용자 풀 사용
import boto3

gateway_client = boto3.client("bedrock-agentcore-control", region_name=os.environ["AWS_DEFAULT_REGION"])
auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [
            client_id
        ],  # 클라이언트는 Cognito에 구성된 ClientId와 반드시 일치해야 함. 예: 7rfbikfsm51j2fpaggacgng84g
        "discoveryUrl": cognito_discovery_url,
    }
}
create_response = gateway_client.create_gateway(
    name="DemoGWOpenAPIAPIKeyNasaOAI",
    roleArn=agentcore_gateway_iam_role["Role"][
        "Arn"
    ],  # IAM 역할에는 Gateway를 생성/나열/조회/삭제할 권한이 있어야 함
    protocolType="MCP",
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration=auth_config,
    description="AgentCore Gateway with OpenAPI target",
)
print(create_response)
# GatewayTarget 생성에 사용할 GatewayID 조회
gatewayID = create_response["gatewayId"]
gatewayURL = create_response["gatewayUrl"]
print(gatewayID)

# Bedrock AgentCore Gateway를 사용하여 NASA Open APIs를 MCP 도구로 변환

NASA Open APIs에서 날씨 데이터를 가져오는 화성 기상 에이전트를 구성합니다. NASA Insight API는 [여기](https://api.nasa.gov/)에서 무료로 등록할 수 있습니다. 등록을 완료하면 이메일로 API 키를 받게 됩니다. 이 API 키를 사용하여 OpenAPI 대상 생성에 필요한 자격 증명 공급자를 구성합니다.

In [ ]:
import boto3
from pprint import pprint

acps = boto3.client(service_name="bedrock-agentcore-control")

response = acps.create_api_key_credential_provider(
    name="NasaInsightAPIKey",
    apiKey="",  # api.nasa.gov에서 가입하여 API 키를 발급받음. 약 2분 후 이메일로 API 키가 전송됨.
)

pprint(response)
credentialProviderARN = response["credentialProviderArn"]
pprint(f"Egress Credentials provider ARN, {credentialProviderARN}")

# OpenAPI 대상 생성

#### NASA Open API json 파일을 S3에 업로드

In [ ]:
# S3 클라이언트 생성
session = boto3.session.Session()
s3_client = session.client("s3")
sts_client = session.client("sts")

# AWS 계정 ID 및 리전 조회
account_id = sts_client.get_caller_identity()["Account"]
region = session.region_name
# 파라미터 정의
# OpenAPI json 파일을 업로드할 s3 버킷
bucket_name = f"agentcore-gateway-{account_id}-{region}"
file_path = "openapi-specs/nasa_mars_insights_openapi.json"
object_key = "nasa_mars_insights_openapi.json"
# put_object를 사용하여 파일을 업로드하고 응답 확인
try:
    if region == "us-east-1":
        s3bucket = s3_client.create_bucket(Bucket=bucket_name)
    else:
        s3bucket = s3_client.create_bucket(Bucket=bucket_name, CreateBucketConfiguration={"LocationConstraint": region})
    with open(file_path, "rb") as file_data:
        response = s3_client.put_object(Bucket=bucket_name, Key=object_key, Body=file_data)

    # 계정 ID와 리전을 사용하여 업로드한 객체의 ARN 구성
    openapi_s3_uri = f"s3://{bucket_name}/{object_key}"
    print(f"Uploaded object S3 URI: {openapi_s3_uri}")
except Exception as e:
    print(f"Error uploading file: {e}")

#### 아웃바운드 인증 구성 및 Gateway 대상 생성

In [ ]:
# OpenAPI 사양 파일의 S3 Uri
nasa_openapi_s3_target_config = {"mcp": {"openApiSchema": {"s3": {"uri": openapi_s3_uri}}}}

# API 키 자격 증명 공급자 구성
api_key_credential_config = [
    {
        "credentialProviderType": "API_KEY",
        "credentialProvider": {
            "apiKeyCredentialProvider": {
                "credentialParameterName": "api_key",  # 각 API 공급자가 요구하는 API 키 이름으로 변경. 헤더에 토큰을 전달하려면 "Authorization" 사용
                "providerArn": credentialProviderARN,
                "credentialLocation": "QUERY_PARAMETER",  # API 키 위치. 가능한 값은 "HEADER" 및 "QUERY_PARAMETER"임.
                # "credentialPrefix": " " # token 접두사입니다. 유효한 값은 "Basic"이며 token에만 적용됩니다.
            }
        },
    }
]

targetname = "DemoOpenAPITargetS3NasaMars"
response = gateway_client.create_gateway_target(
    gatewayIdentifier=gatewayID,
    name=targetname,
    description="OpenAPI Target with S3Uri using SDK",
    targetConfiguration=nasa_openapi_s3_target_config,
    credentialProviderConfigurations=api_key_credential_config,
)

# Strands Agent에서 Bedrock AgentCore Gateway 호출

Strands Agent는 Model Context Protocol(MCP) 사양을 구현하는 Bedrock AgentCore Gateway를 통해 AWS 도구와 원활하게 통합됩니다. 이 통합을 통해 AI 에이전트와 AWS 서비스 간에 안전하고 표준화된 통신이 가능합니다.

Bedrock AgentCore Gateway는 기본 MCP API인 ListTools 및 InvokeTools를 제공하는 프로토콜 호환 Gateway 역할을 합니다. 이러한 API를 통해 모든 MCP 호환 클라이언트 또는 SDK가 안전하고 표준화된 방식으로 사용 가능한 도구를 검색하고 상호 작용할 수 있습니다. Strands Agent가 AWS 서비스에 액세스해야 할 때는 MCP 표준화 엔드포인트를 사용하여 Gateway와 통신합니다.

Gateway 구현은 (MCP 권한 부여 사양)[https://modelcontextprotocol.org/specification/draft/basic/authorization]을 엄격하게 준수하여 강력한 보안과 액세스 제어를 보장합니다. 따라서 Strands Agent의 모든 도구 호출은 권한 부여 단계를 거치며, 보안을 유지하면서 강력한 기능을 사용할 수 있습니다.

예를 들어 Strands Agent가 MCP 도구에 액세스해야 하는 경우 먼저 ListTools를 호출하여 사용 가능한 도구를 검색한 다음 InvokeTools를 사용하여 특정 작업을 실행합니다. Gateway는 필요한 모든 보안 검증, 프로토콜 변환 및 서비스 상호 작용을 처리하므로 전체 과정이 원활하고 안전하게 진행됩니다.

이 아키텍처를 사용하면 MCP 사양을 구현하는 모든 클라이언트 또는 SDK가 Gateway를 통해 AWS 서비스와 상호 작용할 수 있으므로, AI 에이전트 통합을 위한 범용적이고 미래 지향적인 솔루션을 구축할 수 있습니다.

# 인바운드 권한 부여를 위해 Amazon Cognito에 액세스 토큰 요청

In [ ]:
print(
    "Requesting the access token from Amazon Cognito authorizer...May fail for some time till the domain name propogation completes"
)
token_response = utils.get_token(user_pool_id, client_id, client_secret, scopeString, REGION)
token = token_response["access_token"]
print("Token response:", token)

# Bedrock AgentCore Gateway로 NASA Open APIs를 호출하여 화성 기상 에이전트에 질문

In [ ]:
from strands.models import BedrockModel
from mcp.client.streamable_http import streamablehttp_client
from strands.tools.mcp.mcp_client import MCPClient
from strands import Agent


def create_streamable_http_transport():
    return streamablehttp_client(gatewayURL, headers={"Authorization": f"Bearer {token}"})


client = MCPClient(create_streamable_http_transport)

## ~/.aws/credentials에 구성된 IAM 그룹/사용자에는 Bedrock 모델 액세스 권한이 있어야 함
yourmodel = BedrockModel(
    model_id="us.amazon.nova-pro-v1:0",
    temperature=0.7,
)

In [ ]:
import logging


# 루트 strands 로거 구성. 문제를 디버깅하는 경우 DEBUG로 변경
logging.getLogger("strands").setLevel(logging.INFO)

# 로그를 확인할 핸들러 추가
logging.basicConfig(format="%(levelname)s | %(name)s | %(message)s", handlers=[logging.StreamHandler()])

with client:
    # listTools 호출
    tools = client.list_tools_sync()
    # 모델과 도구를 사용하여 Agent 생성
    agent = Agent(model=yourmodel, tools=tools)  ## 원하는 모델로 교체 가능
    print(f"Tools loaded in the agent are {agent.tool_names}")
    # print(f"Tools configuration in the agent are {agent.tool_config}")
    # 샘플 프롬프트로 에이전트 호출. MCP listTools만 호출하여 LLM이 액세스할 수 있는 도구 목록을 조회하며, 아래에서는 실제 도구를 호출하지 않음
    agent("Hi , can you list all tools available to you")
    agent("What is the weather in northern part of the mars")
    # 샘플 프롬프트로 에이전트를 호출하고 도구를 실행한 후 응답 표시
    # MCP 도구를 명시적으로 호출. MCP 도구 이름과 인수는 AWS Lambda 함수 또는 OpenAPI/Smithy API와 일치해야 함
    result = client.call_tool_sync(
        tool_use_id="get-insight-weather-1",  # 고유 식별자로 교체 가능
        name=targetname
        + "___getInsightWeather",  # AWS Lambda 대상 유형을 기준으로 한 도구 이름이며 대상 이름에 따라 변경됨
        arguments={"ver": "1.0", "feedtype": "json"},
    )
    # MCP 도구 응답 출력
    print(f"Tool Call result: {result['content'][0]['text']}")

**문제: 아래 셀을 실행할 때 다음 오류가 발생하면 pydantic과 pydantic-core 버전이 호환되지 않는 것입니다.**

```
TypeError: model_schema() got an unexpected keyword argument 'generic_origin'
```
**해결 방법**

서로 호환되는 pydantic==2.7.2와 pydantic-core 2.27.2가 설치되어 있는지 확인해야 합니다. 설치가 완료되면 커널을 다시 시작합니다.

# 정리
IAM 역할, IAM 정책, 자격 증명 공급자, AWS Lambda 함수, Cognito 사용자 풀, s3 버킷과 같은 추가 리소스도 생성됩니다. 정리 과정에서 이러한 리소스를 수동으로 삭제해야 할 수 있으며, 실행한 예제에 따라 달라집니다.

## Gateway 삭제(선택 사항)

In [ ]:
import utils

utils.delete_gateway(gateway_client, gatewayID)